# Level 1 — Classic CNNs (VGG-16, ResNet-18 / 50)

**목표**: VGG 와 ResNet 을 직접 구현하여 Set A 위에서 Multi-task 로 학습하고, **Skip Connection** 이 깊은 네트워크의 수렴에 미치는 영향을 분석합니다.

**금지 사항**: `torchvision.models.*`, `timm`. 단, 위 라이브러리의 코드를 *참고하여 직접 타이핑* 하는 것은 허용됩니다.

본 노트북의 산출물:
1. 학습된 체크포인트 (`checkpoints/level1_*.pth`).
2. VGG-16, ResNet-18, ResNet-50 의 `Avg-Macro-F1` 및 속성별 Macro-F1 표.
3. VGG (skip 없음) vs ResNet (skip 있음) 의 학습 손실 곡선 비교.

In [ ]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/Leem1nwon/2026-HYU-AUE8088-PA2.git

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

import torch
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.resnet import resnet18, resnet50
from src.models.vgg import VGG16

SEED = 42
set_seed(SEED, deterministic=True)   # 재현성을 위한 시드 고정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 31.2 MB/s eta 0:00:00
device: cuda


### Weights & Biases 설정

학습 곡선과 속성별 Macro-F1 을 자동으로 로깅합니다. 사용하지 않으려면 `WANDB_PROJECT = None` 으로 두세요 — 로깅 호출은 자동으로 no-op 이 됩니다.

최초 1회만:
```python
import wandb; wandb.login()   # API key 입력
```

In [ ]:
# Colab `Run All` 재현 시 wandb 는 비활성화합니다 (조교 채점 환경 = WANDB_DISABLED=true).
# 본인이 직접 학습하며 로깅하려면 이 줄을 주석 처리하고 아래 셀에서 RUN_TRAINING=True, wandb.login() 을 사용하세요.
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# ===== 재현 모드 스위치 =====
# RUN_TRAINING = False → 학습을 건너뛰고 사전학습 체크포인트(.pth)를 로드해 eval/그림만 재현 (조교 채점 경로).
# RUN_TRAINING = True  → 처음부터 학습 (H100 권장; T4 에서는 매우 오래 걸립니다).
RUN_TRAINING = False

WANDB_PROJECT = "aue8088-pa2" if RUN_TRAINING else None   # 재현 모드에서는 wandb 끔
WANDB_TAGS    = ["level1"]

In [ ]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

# Set A 의 train/val 로더 구성
train_ds = BDDAttrDataset(DATA_ROOT, split="train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, split="val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

# 속성별 클래스 분포 출력 — 불균형 정도를 직접 확인할 것
for a in ATTRIBUTES:
    print(f"{a:10s} 클래스별 샘플 수 (train) = {train_ds.class_counts(a).tolist()}")

In [ ]:
# --- 사전학습 체크포인트 다운로드 (재현 모드 RUN_TRAINING=False) ----------------
# checkpoints/ 에 .pth 가 없으면 Google Drive 에서 체크포인트 zip 을 받아 압축 해제.
# (제출 전 체크포인트 zip 을 본인 드라이브에 올린 뒤 아래 ID 를 교체하세요.)
CKPT_DIR = "../checkpoints"
GDRIVE_CKPT_ID = "PUT_CHECKPOINT_ZIP_ID_HERE"   # ← 체크포인트 zip 의 Google Drive 파일 ID

if not RUN_TRAINING and not os.path.exists(f"{CKPT_DIR}/level1_resnet18.pth"):
    if GDRIVE_CKPT_ID == "PUT_CHECKPOINT_ZIP_ID_HERE":
        print("[주의] GDRIVE_CKPT_ID 를 체크포인트 zip 파일 ID 로 교체하세요.")
    else:
        import gdown
        gdown.download(id=GDRIVE_CKPT_ID, output="../checkpoints.zip", quiet=False)
        with zipfile.ZipFile("../checkpoints.zip") as zf:
            zf.extractall("..")
        print("체크포인트 다운로드/해제 완료")
else:
    print(f"체크포인트 준비 상태: RUN_TRAINING={RUN_TRAINING}")

In [ ]:
from torch import nn

def train_one(model_fn, name, epochs=30, loss_weights=None):
    """단일 모델을 학습하고 체크포인트(fp32)와 wandb 산출물을 저장한다."""
    set_seed(SEED, deterministic=True)
    model = model_fn().to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    cfg = TrainConfig(epochs=epochs,
                      loss_weights=loss_weights or {"weather": 1.0, "scene": 1.0, "timeofday": 1.0})
    logger = WandbLogger(
        project=WANDB_PROJECT, run_name=f"level1-{name}",
        config={"backbone": name, "epochs": epochs, "batch": BATCH,
                "lr": 3e-4, "weight_decay": 5e-4, "seed": SEED, "loss_weights": cfg.loss_weights},
        tags=WANDB_TAGS + [name])
    trainer = MultiTaskTrainer(model, optim, sched, losses, device, cfg, logger=logger)
    history = trainer.fit(train_loader, val_loader)
    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", confusion_matrices(val_pred, val_tgt)[a], CLASS_NAMES[a])
    logger.finish()
    os.makedirs(CKPT_DIR, exist_ok=True)
    torch.save({"state_dict": model.float().state_dict(), "history": history},
               f"{CKPT_DIR}/level1_{name}.pth")
    return model, history

def load_model(model_fn, name):
    """사전학습 체크포인트 로드 (재현 모드)."""
    model = model_fn().to(device)
    ckpt = torch.load(f"{CKPT_DIR}/level1_{name}.pth", map_location="cpu")
    model.load_state_dict(ckpt["state_dict"])
    model.to(device).eval()
    return model, ckpt.get("history")

# VGG16(skip 없음) / ResNet-18 / ResNet-50(skip 있음) — RUN_TRAINING 에 따라 학습 또는 로드
MODELS = {"vgg16": VGG16, "resnet18": resnet18, "resnet50": resnet50}
results = {}
for name, fn in MODELS.items():
    model, history = (train_one(fn, name, epochs=30) if RUN_TRAINING else load_model(fn, name))
    results[name] = {"model": model, "history": history}
    print(f"[{name}] {'학습 완료' if RUN_TRAINING else '체크포인트 로드'}")

## 분석 (리포트 필수 포함 항목)

1. **수렴 비교**: VGG-16 과 ResNet-18 의 학습 손실 곡선을 함께 그리세요. Skip connection 이 없는 깊은 네트워크가 정체되는 현상이 관찰되는지, 그 원인은 무엇인지 서술하세요.
2. **속성별 MF1 표**: 각 백본에 대해 Weather / Scene / Time of Day 의 Macro-F1 을 표로 정리하세요. 어느 속성이 가장 어려운지, 깊이 변화에 따라 그 양상이 어떻게 바뀌는지 분석하세요.
3. **Loss 가중치 민감도**: ResNet-18 에 대해 최소 두 가지 비자명한 가중치 설정 (예: `(1, 1, 1)` vs `(2, 1, 1)`) 으로 재학습하고 Avg-MF1 변화를 보고하세요.

wandb 를 활성화한 경우 같은 프로젝트의 Run 들을 비교하면 학습 곡선·속성별 MF1·confusion matrix 가 한눈에 정리됩니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.utils.metrics import average_macro_f1, per_attribute_macro_f1, per_class_prf

# 1) Backbone 비교 — Avg-MF1 / 속성별 Macro-F1 (val)
print(f"{'backbone':10s} {'skip':>5s} {'Avg-MF1':>8s}  weather  scene  timeofday")
preds_cache = {}
for name in MODELS:
    m = results[name]["model"]
    preds, _, tgts, _ = collect_predictions(m, val_loader, device)
    preds_cache[name] = (preds, tgts)
    avg = average_macro_f1(preds, tgts); per = per_attribute_macro_f1(preds, tgts)
    skip = "no" if name == "vgg16" else "yes"
    print(f"{name:10s} {skip:>5s} {avg:8.4f}  {per['weather']:.3f}  {per['scene']:.3f}  {per['timeofday']:.3f}")

# 2) Skip connection 효과 — VGG(no-skip) vs ResNet(skip) train loss 곡선
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name in MODELS:
    h = results[name]["history"]
    if h and "train_loss" in h:
        ax[0].plot(h["train_loss"], label=name, ls="--" if name == "vgg16" else "-")
        ax[1].plot(h["val_avg_mf1"], label=name, ls="--" if name == "vgg16" else "-")
ax[0].set_title("Train loss (skip effect)"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
ax[1].set_title("Val Avg-MF1"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.show()

# 3) best backbone(ResNet-18) 속성별 정규화 Confusion Matrix
import seaborn as sns
preds, tgts = preds_cache["resnet18"]
cms = confusion_matrices(preds, tgts)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax_, a in zip(axes, ATTRIBUTES):
    sns.heatmap(cms[a], annot=True, fmt=".2f", cmap="Blues", cbar=False,
                xticklabels=CLASS_NAMES[a], yticklabels=CLASS_NAMES[a], ax=ax_)
    ax_.set_title(f"ResNet-18 — {a}"); ax_.set_xlabel("pred"); ax_.set_ylabel("true")
plt.tight_layout(); plt.show()